In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, min, max, mean, when ,sum
from pyspark.sql.types import TimestampType

spark = SparkSession.builder.appName("Week_05_Assignment").getOrCreate()

df = spark.read.csv("Spark_assignment_dataset.csv", header=True, inferSchema=True)

df.printSchema()      # shows column names and types
df.show(5)         # shows first 5 rows
print("Total rows:", df.count())


root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- store_id: integer (nullable = true)

+-------+----------------+------+----------------+-----------+-------+---+------------+-------------------+----------------+--------+------+---------+--------+
|user_id|transaction_date|region|product_category|sale_amount|   city|age|subscription|      raw_timestamp|           email|username| price|   status|store_id|
+-------+----------------+------+----------------+-----------+-------+---+------------+---------------

-------------------

### Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing? 

Traditional MapReduce has a few major limitations. The biggest one is that it writes intermediate data to disk after every map and reduce step. Since reading and writing to disk is slow, this creates a performance bottleneck, especially for jobs with many steps. MapReduce is also hard to program because every problem has to be forced into a rigid map-and-reduce structure, which makes complex logic awkward. It also performs poorly for iterative tasks like machine learning, where the same data is processed again and again, because each iteration reloads data from disk. Spark solves these issues by keeping data in memory, offering a much simpler API (DataFrames, SQL), and supporting iterative processing efficiently. This is why Spark is generally preferred for modern big data work.

--------------

### Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

In disk-based systems like MapReduce, every iteration of an algorithm reads its input from disk and writes its output back to disk. Machine learning algorithms (like gradient descent or k-means) repeat the same computation many times over the same dataset, so this repeated disk access adds up to a huge cost. Spark instead loads the dataset into RAM once and keeps it there across iterations. Because memory access is far faster than disk access, each iteration runs much quicker. Spark can cache an RDD or DataFrame in memory using .cache() or .persist(), so the data doesn't have to be recomputed or reloaded. This is why Spark can be up to 100x faster than MapReduce for iterative ML algorithms.

--------------------

### Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.

In [3]:
df_Q3 = df.dropDuplicates(["user_id", "transaction_date"])

print("Rows before:", df.count())
print("Rows after dropDuplicates:", df_Q3.count())
print("Duplicates removed:", df.count() - df_Q3.count())


Rows before: 1220
Rows after dropDuplicates: 1191
Duplicates removed: 29


----------------

### Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.

In [7]:
df_Q4 = df.filter(col("region") == "West") \
          .groupBy("product_category") \
          .agg(avg("sale_amount").alias("avg_sale_amount"))

df_Q4.show()

+----------------+------------------+
|product_category|   avg_sale_amount|
+----------------+------------------+
|         Grocery| 485.9641666666667|
|     Electronics|487.74397058823524|
|        Clothing| 548.1489655172413|
|           Books|  510.405909090909|
|       Furniture| 525.3246774193549|
+----------------+------------------+



This filters to West region first, then groups by product category and computes the average sale amount per category.

---------------------

### Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'. 

.na.drop() removes entire rows that contain null values, while .na.fill() keeps the rows but replaces the null values with a value you specify. You use drop when missing data makes a row unusable, and fill when you'd rather keep the row and substitute a sensible default.

In [8]:
df_Q5 = df.na.fill("Unknown", subset=["status"])
df_Q5.select("status").show(10)

+---------+
|   status|
+---------+
|  Pending|
|  Pending|
|Cancelled|
|  Unknown|
|  Pending|
|Completed|
|Cancelled|
|  Pending|
|  Pending|
|  Unknown|
+---------+
only showing top 10 rows


----------------------------

### Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In [9]:
df_Q6 = df.groupBy("city") \
          .agg(count("*").alias("record_count")) \
          .filter(col("record_count") > 100)

df_Q6.show()


+---------+------------+
|     city|record_count|
+---------+------------+
|  Chennai|         117|
|   Mumbai|         175|
|     Pune|         157|
|    Delhi|         213|
|Bengaluru|         137|
|   Jaipur|         262|
+---------+------------+



Group by city, count records in each, then filter to keep only cities with more than 100.

------------------

### Q7: How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them?

Spark DataFrames are immutable, meaning they can't be changed in place. When you drop or rename a column, Spark doesn't modify the existing DataFrameit creates a brand new one with the change applied. Because of this, you have to reassign the result to a variable, for example df = df.drop("col") or df = df.withColumnRenamed("old", "new"). If you forget to reassign, your change is lost because the original DataFrame stays exactly as it was. Immutability makes Spark reliable and fault-tolerant, since the original data is never accidentally corrupted.

------------------------

### Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

In [12]:
df_Q8 = df.filter((col("age") > 18) & (col("age") <= 30) &
                  (col("subscription") == "Premium"))

df_Q8.select("user_id", "age", "subscription").show(10)

+-------+---+------------+
|user_id|age|subscription|
+-------+---+------------+
|    273| 27|     Premium|
|    283| 22|     Premium|
|    290| 30|     Premium|
|     80| 23|     Premium|
|    161| 26|     Premium|
|    135| 20|     Premium|
|    272| 23|     Premium|
|     81| 24|     Premium|
|     71| 29|     Premium|
|     46| 24|     Premium|
+-------+---+------------+
only showing top 10 rows


-----------------------

### Q9: When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()? 

It's better to handle nulls first because aggregation functions can behave unexpectedly with missing data. For example, avg() ignores nulls when calculating the mean, which can quietly skew the result if a lot of values are missing. Counts can also be misleading depending on whether nulls are included. By deciding upfront whether to drop or fill nulls, you make sure the aggregation is based on clean, consistent data and the results are accurate and reproducible rather than silently distorted.

-----------------------------

### Q10: Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

In [13]:
df_Q10 = df.withColumn("raw_timestamp", col("raw_timestamp").cast(TimestampType())) \
           .withColumnRenamed("raw_timestamp", "event_time")

df_Q10.printSchema()
df_Q10.select("event_time").show(5)


root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- store_id: integer (nullable = true)

+-------------------+
|         event_time|
+-------------------+
|2025-02-19 23:33:00|
|2025-05-15 17:54:00|
|2025-03-28 01:35:00|
|2025-05-14 18:02:00|
|2025-05-29 01:06:00|
+-------------------+
only showing top 5 rows


In this particular dataset raw_timestamp was already stored in a clean yyyy-MM-dd HH:mm:ss format, so when I loaded the file with inferSchema=True, Spark had already auto-detected it as a timestamp column on its own, and the cast above worked for all 1220 rows with 0 rows turning null. If the source data had messier/inconsistent formats, some rows would silently turn into null after the cast instead.

-----------------

### Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

When you do a groupBy, Spark needs all the records that share the same key (for example, the same city) to be processed together. But those records may be scattered across many machines in the cluster. The Shuffle is the process where Spark redistributes the data across the network so that all rows with the same key end up on the same partition. This is why grouping is called a wide transformation it requires data to move between partitions (a "wide" dependency), unlike a narrow transformation such as filter where each partition can be handled independently. The shuffle is expensive because it involves network transfer and disk I/O, so it's usually the slowest part of a Spark job.

-------------------------

### Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.

In [17]:
df_Q12 = df.filter(col("email").isNotNull() & (col("username") != ""))

print("Before:", df.count())
print("After:", df_Q12.count())


Before: 1220
After: 1057


This keeps only rows where email is not null AND username is not empty which is the same as removing rows where email is null OR username is empty.

------------------

### Q13: How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?

In [18]:
df_Q13 = df.agg(
    min("price").alias("min_price"),
    max("price").alias("max_price"),
    mean("price").alias("mean_price")
)

df_Q13.show()


+---------+---------+-----------------+
|min_price|max_price|       mean_price|
+---------+---------+-----------------+
|      5.5|   499.96|257.7034919210054|
+---------+---------+-----------------+



.agg() lets you compute several aggregations in one call by passing multiple functions. Record the values.

------------------------

### Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats? 

When inferSchema=true, Spark scans the data and guesses each column's type automatically. If a date column has inconsistent formats (like mixing 2023-01-05, 05/01/2023, and 01-05-2023), Spark can't recognize a single consistent date pattern, so it usually gives up and reads the column as a plain string instead of a proper date type. This means you lose date functionality and may get wrong results later when sorting or filtering by date. It can also misinterpret some values, leading to nulls or incorrect parsing. The safer approach is to define the schema manually or read dates as strings and parse them explicitly with a known format.

-------------

### Q15: Write a final processing pipeline that: 

#### Filters out duplicates. 

#### Fills null prices with 0. 

#### Groups by store_id to calculate total revenue. 

In [19]:
df_Q15 = df.dropDuplicates() \
           .na.fill(0, subset=["price"]) \
           .groupBy("store_id") \
           .agg(sum("price").alias("total_revenue"))

df_Q15.show()


+--------+------------------+
|store_id|     total_revenue|
+--------+------------------+
|     101|          53010.96|
|     103| 56895.36000000002|
|     102|52718.260000000024|
|     105| 58738.08999999998|
|     104| 60837.63999999996|
+--------+------------------+



This pipeline removes duplicate rows, fills any null prices with 0, then groups by store_id and sums the price to get total revenue per store.



----------------